#Lead Data Engineer Task
#PEI Group
**Scenario: E-commerce Sales Data Processing with Databricks You've been assigned to design and implement a data processing system using Databricks for an e-commerce platform. This platform generates a lot of sales data, including details about orders, products, customers, and transactions. Your goal is to use Databricks to create a scalable, efficient, and reliable data engineering solution. This solution should process and analyze the data to provide valuable insights for business stakeholders**

In [0]:
%python %pip install openpyxl

In [0]:
%restart_python

In [0]:
from pyspark.sql.functions import *
import re

json_path = "/Volumes/workspace/default/pei/Orders.json"

df_orders = spark.read.format("json").option("multiline","true").load(json_path)

# Delta tables disallow ' ,;{}()\n\t= in column names, so sanitize them (e.g. "Customer ID" -> "Customer_ID")
df_orders = df_orders.toDF(*[re.sub(r"[ ,;{}()\n\t=]", "_", c) for c in df_orders.columns])

#display(df_orders)

df_orders.write.format("delta").mode("overwrite").saveAsTable("filter_order")

In [0]:
import pandas as pd
import re

excel_path = "/Volumes/workspace/default/pei/Customer.xlsx"

# Read Excel file
%pip install openpyxl
pdf = pd.read_excel(excel_path, engine="openpyxl").astype({'phone': 'str'})

# Convert Pandas DataFrame to Spark DataFrame
df_customer = spark.createDataFrame(pdf)

# Delta tables disallow ' ,;{}()\n\t= in column names, so sanitize them (e.g. "Customer ID" -> "Customer_ID")
df_customer = df_customer.toDF(*[re.sub(r"[ ,;{}()\n\t=]", "_", c) for c in df_customer.columns])

#display(df_customer)

#Create Raw Delta table
df_customer.write.format("delta").mode("overwrite").saveAsTable("filter_customer")

In [0]:
csv_path = "/Volumes/workspace/default/pei/Products.csv"

import re

df_products = spark.read.format("csv").option("header", "true").option("inferSchema", "true") .option("delimiter", ",").load(csv_path)

# Delta tables disallow ' ,;{}()\n\t= in column names, so sanitize them (e.g. "Product ID" -> "Product_ID")
df_products = df_products.toDF(*[re.sub(r"[ ,;{}()\n\t=]", "_", c) for c in df_products.columns])

#display(df_products)

df_products.write.format("delta").mode("overwrite").saveAsTable("filter_product")

In [0]:
df_prod_cust = df_customer.join(df_products,on="State",how="inner")

df_prod_cust.write.format("delta").mode("overwrite").saveAsTable("enriched_prod_cust")

#display(df_prod_cust)

In [0]:
from pyspark.sql.functions import col, round

# Rebuild df_enriched with correct join keys and column names
# (orders -> customers on Customer_ID, -> products on Product_ID)
df_enriched = (
    df_orders
    .join(df_customer, "Customer_ID", "inner")
    .join(df_products, "Product_ID", "inner")
    .select(
        col("Order_ID"),
        col("Order_Date"),
        round(col("Profit"), 2).alias("Profit"),
        col("Customer_Name"),
        col("Country"),
        col("Category"),
        col("Sub-Category")
    )
)

df_enriched.write.format("delta").mode("overwrite").saveAsTable("final_enriched_sales")

In [0]:
from pyspark.sql.functions import year, sum, col, to_date

df_aggregate = (
    df_enriched
    .withColumn("Year", year(to_date(col("Order_Date"), "d/M/yyyy")))
    .groupBy(
        "Year",
        "Category",
        "Sub-Category",
        "Customer_Name"
    )
    .agg(
        round(sum("Profit"), 2).alias("Total_Profit")
    )
)

df_aggregate.write.format("delta").mode("overwrite").saveAsTable("agg_profit")

#Test cases of first 4 problems

In [0]:
from pyspark.sql.functions import year, sum, col, to_date

# Test-case 1: No null check on OrderId
assert df_enriched.filter(
    col("Order_ID").isNull()
).count() == 0

print("no null check for order_id done")

# Test-case 2: Profit should not be null
assert df_enriched.filter(
    col("Profit").isNull()
).count() == 0

print("not null check done")

# Test-case 3: Profit should have maximum 2 decimal places
assert df_enriched.filter(
    col("Profit") != round(col("Profit"), 2)
).count() == 0

print("profit check decimal upto 2 digits is done")

# Test-case 4: Aggregate profit should match enriched sales profit
total_enriched = df_enriched.agg(
    sum("Profit")
).first()[0]

total_aggregate = df_aggregate.agg(
    sum("Total_Profit")
).first()[0]

print("total aggregation is done")

# Test-case 5: schema  check - if all columns exists
required_columns = [
        "Order_ID",
        "Order_Date",
        "Customer_Name",
        "Country",
        "Category",
        "Sub-Category"
        ]
columns_check = df_enriched

for c in required_columns:
    assert c in required_columns, f"Missing column: {c}"

print("schema check done")

print("All test cases passed!")

**We required only encriched table as it is having all transformed data**

In [0]:
%sql
select * from final_enriched_sales limit 10 ;

In [0]:
%sql
-- Profit By Year
SELECT
    YEAR(to_date(Order_Date, 'd/M/yyyy')) AS Year,
    ROUND(SUM(Profit), 2) AS Total_Profit
FROM final_enriched_sales
GROUP BY YEAR(to_date(Order_Date, 'd/M/yyyy'))
ORDER BY Year;

-- Profit By Year and Product Category
SELECT
    YEAR(to_date(Order_Date, 'd/M/yyyy')) AS Year,
    Category,
    ROUND(SUM(Profit), 2) AS Total_Profit
FROM final_enriched_sales
GROUP BY YEAR(to_date(Order_Date, 'd/M/yyyy')), Category
ORDER BY Year, Category;

-- Profit By Customer
SELECT
    Customer_Name,
    ROUND(SUM(Profit), 2) AS Total_Profit
FROM final_enriched_sales
GROUP BY Customer_Name
ORDER BY Total_Profit DESC;

-- Profit By Customer + Year
SELECT
    Customer_Name,
    YEAR(to_date(Order_Date, 'd/M/yyyy')) AS Year,
    ROUND(SUM(Profit), 2) AS Total_Profit
FROM final_enriched_sales
GROUP BY Customer_Name, YEAR(to_date(Order_Date, 'd/M/yyyy'))
ORDER BY Customer_Name, Year;